# 04 · Comparative Summary

This notebook synthesises results from notebooks 02 and 03 into a single cross-metric
comparison suitable for the thesis results chapter.

**Units:**
- Energy → **J** (Joules)
- Time → **s** (seconds)
- EDP → **J·s** (Joule-seconds)

**Outputs:**
- Normalised efficiency ranking table (ratio vs best language)
- Top 3 / Bottom 3 per metric
- Key findings bullet list
- `outputs/per_language_correlations.csv` — per-language Spearman table

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
COMPILER        = ps.COMPILER
COMPILER_COLORS = ps.COMPILER_COLORS
COMPILER_ORDER  = ps.COMPILER_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Normalized Efficiency Comparison

Each metric is expressed as a **ratio relative to the best (lowest) language** — so 1.0 = best
and e.g. 5.0 means this language consumes 5× more CPU energy / RAM energy / time than the
most efficient language. Values are computed from the **mean** across all 8 benchmarks,
matching the CLBG presentation style.

In [ ]:
# Two-step mean (equal benchmark weight) straight from the per-cell means in df_mean.
norm_agg = lang_means([COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME])

norm_agg['CPU Energy Normalized'] = (norm_agg[COL_CPU_ENERGY] / norm_agg[COL_CPU_ENERGY].min()).round(2)
norm_agg['RAM Energy Normalized'] = (norm_agg[COL_MEM_ENERGY] / norm_agg[COL_MEM_ENERGY].min()).round(2)
norm_agg['Time Normalized']       = (norm_agg[COL_TIME]        / norm_agg[COL_TIME].min()).round(2)

norm_ranking = (
    norm_agg[['CPU Energy Normalized', 'RAM Energy Normalized', 'Time Normalized']]
    .sort_values('CPU Energy Normalized')
    .reset_index()
    .rename(columns={'language': 'Language'})
)

norm_ranking.index = norm_ranking.index + 1  # rank starts at 1
norm_ranking.index.name = 'Rank'

norm_ranking

In [ ]:
# Pretty table figure of the normalised ranking, with an Execution Model column and
# rows tinted by compiler.
import matplotlib.colors as mcolors

tbl = norm_ranking.copy()
tbl.insert(1, 'Execution Model', tbl['Language'].map(COMPILER))

def _tint(hexc, amt=0.82):
    """Mix a hex colour toward white by `amt` (0=full colour, 1=white)."""
    r, g, b = mcolors.to_rgb(hexc)
    return (r + (1 - r) * amt, g + (1 - g) * amt, b + (1 - b) * amt)

col_labels = ['Rank', 'Language', 'Execution Model', 'CPU Energy (×)', 'RAM Energy (×)', 'Time (×)']
cell_text = []
for rank in tbl.index:
    r = tbl.loc[rank]
    cell_text.append([str(rank), r['Language'], r['Execution Model'],
                      f"{r['CPU Energy Normalized']:.2f}",
                      f"{r['RAM Energy Normalized']:.2f}",
                      f"{r['Time Normalized']:.2f}"])

fig, ax = plt.subplots(figsize=(11, 8))
ax.axis('off')
table = ax.table(cellText=cell_text, colLabels=col_labels, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

# header
for j in range(len(col_labels)):
    h = table[(0, j)]
    h.set_facecolor('#2c3e50')
    h.set_text_props(color='white', fontweight='bold')

# data rows: tint whole row by compiler, stronger tint on the Execution Model cell
for i, rank in enumerate(tbl.index, start=1):
    base = COMPILER_COLORS[tbl.loc[rank, 'Execution Model']]
    for j in range(len(col_labels)):
        table[(i, j)].set_facecolor(_tint(base, 0.82))
    table[(i, 1)].set_text_props(fontweight='bold')      # Language
    table[(i, 2)].set_facecolor(_tint(base, 0.55))       # Execution Model chip

# column widths (sum = 1.0)
widths = [0.08, 0.22, 0.20, 0.18, 0.18, 0.16]
for j, w in enumerate(widths):
    for i in range(len(tbl) + 1):
        table[(i, j)].set_width(w)

legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                  for p in COMPILER_ORDER]
ax.legend(handles=legend_handles, title='Execution Model', loc='lower center',
          bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=True)
ax.set_title('Normalised Efficiency Ranking — ratio vs best language (1.0 = best)\n'
             'sorted by CPU energy', fontsize=12, pad=18)
plt.tight_layout()
ps.save_fig(fig, '04_normalised_ranking_table')
plt.show()

> **Takeaway:** the multiplier view makes the gap concrete — the worst languages use ~80× the CPU energy and ~60× the time of the best. AOT languages stay in single-digit multiples; the JIT/interpreted languages blow out by 1–2 orders of magnitude.

## 2. Top 3 / Bottom 3 per Metric

Quick-reference tables for the most and least efficient languages on each metric.
All values in human-readable units (J, s, J·s).

In [ ]:
metric_cols = {
    'CPU Energy (J)':    COL_CPU_ENERGY,
    'Memory Energy (J)': COL_MEM_ENERGY,
    'Execution Time (s)':COL_TIME,
    'EDP (J·s)':         'EDP',
}
units = {'CPU Energy (J)': 'J', 'Memory Energy (J)': 'J',
         'Execution Time (s)': 's', 'EDP (J·s)': 'J·s'}

for label, col in metric_cols.items():
    mean_series = df_mean.groupby('language')[col].mean().sort_values()
    top3 = mean_series.head(3)
    bot3 = mean_series.tail(3)
    unit = units[label]
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"  Top 3 (most efficient):")
    for lang, val in top3.items():
        print(f"    • {lang:12s} ({COMPILER[lang]:12s})  {val:>10.3f} {unit}")
    print(f"  Bottom 3 (least efficient):")
    for lang, val in bot3.items():
        print(f"    • {lang:12s} ({COMPILER[lang]:12s})  {val:>10.3f} {unit}")

## 3. Key Findings

A bullet-point summary of the main findings in human-readable units (J, s, J·s),
suitable for direct citation in the thesis.

In [ ]:
cpu_rank_ser  = df_mean.groupby('language')[COL_CPU_ENERGY].mean().sort_values()
time_rank_ser = df_mean.groupby('language')[COL_TIME].mean().sort_values()
edp_rank_ser  = df_mean.groupby('language')['EDP'].mean().sort_values()

aot_cpu  = df_mean[df_mean['compiler']=='AOT'][COL_CPU_ENERGY].mean()
jit_cpu  = df_mean[df_mean['compiler']=='JIT'][COL_CPU_ENERGY].mean()
int_cpu  = df_mean[df_mean['compiler']=='Interpreted'][COL_CPU_ENERGY].mean()
aot_time = df_mean[df_mean['compiler']=='AOT'][COL_TIME].mean()
jit_time = df_mean[df_mean['compiler']=='JIT'][COL_TIME].mean()
int_time = df_mean[df_mean['compiler']=='Interpreted'][COL_TIME].mean()

print("""
KEY FINDINGS — Benchmark Energy & Time Analysis (18 languages, 8 CLBG benchmarks)
═════════════════════════════════════════════════════════════════════════════════""")

print(f"""
CPU ENERGY (unit: J)
  • Most efficient:   {', '.join(cpu_rank_ser.head(3).index)}
  • Least efficient:  {', '.join(cpu_rank_ser.tail(3).index)}
  • AOT mean:         {aot_cpu:.2f} J
  • JIT mean:         {jit_cpu:.2f} J  ({jit_cpu/aot_cpu:.1f}× AOT)
  • Interpreted mean: {int_cpu:.2f} J  ({int_cpu/aot_cpu:.1f}× AOT)

EXECUTION TIME (unit: s)
  • Fastest:          {', '.join(time_rank_ser.head(3).index)}
  • Slowest:          {', '.join(time_rank_ser.tail(3).index)}
  • AOT mean:         {aot_time:.2f} s
  • JIT mean:         {jit_time:.2f} s  ({jit_time/aot_time:.1f}× AOT)
  • Interpreted mean: {int_time:.2f} s  ({int_time/aot_time:.1f}× AOT)

ENERGY-DELAY PRODUCT (unit: J·s)
  • Best EDP:         {', '.join(edp_rank_ser.head(3).index)}
  • Worst EDP:        {', '.join(edp_rank_ser.tail(3).index)}
  • Best mean EDP:    {edp_rank_ser.iloc[0]:.2f} J·s  ({edp_rank_ser.index[0]})
  • Worst mean EDP:   {edp_rank_ser.iloc[-1]:.2f} J·s ({edp_rank_ser.index[-1]})

Note: All values are two-step means (per-cell mean → mean across 8 CLBG benchmarks,
equal benchmark weight). Use non-parametric tests (notebooks 02–03) for compiler distributions.
""")

## 4. Per-Language Correlation Table

Spearman rank correlation **per language**, computed across that language's **8 benchmark means** (so N = 8 per language) — the format of the CLBG paper's Table 5. "Energy" = CPU energy, "Memory" = DRAM energy, "Time" = execution time. Each ρ is labelled with the **Rea & Parker** effect size on |ρ| (`None` when ρ = 0).

> Note: this is a *different* question from the correlation table in notebook 03, which correlates the 18 per-language means (N = 18). Here each language gets its own ρ, showing whether the energy ↔ time ↔ memory relationship holds *within* that language across workloads.

In [ ]:
# Per-language Spearman correlations across the 8 benchmarks (N=8 per language),
# in the style of the CLBG paper's Table 5. "Energy" = CPU energy, "Memory" = DRAM
# energy, "Time" = execution time. Classification: Rea & Parker on |rho|.
import matplotlib.colors as mcolors

def classify(r):
    """Rea & Parker nominal effect-size label for |rho| ('None' when exactly 0)."""
    a = abs(r)
    if a == 0:   return 'None'
    if a < 0.10: return 'Negligible'
    if a < 0.20: return 'Weak'
    if a < 0.40: return 'Moderate'
    if a < 0.60: return 'Relatively strong'
    if a < 0.80: return 'Strong'
    return 'Very strong'

PAIRS = [
    ('Energy & Time',   COL_CPU_ENERGY, COL_TIME),
    ('Energy & Memory', COL_CPU_ENERGY, COL_MEM_ENERGY),
    ('Memory & Time',   COL_MEM_ENERGY, COL_TIME),
]

records, index = [], []
for lang in ps.LANGUAGE_ORDER:
    sub = df_mean[df_mean['language'] == lang]        # 8 benchmark rows
    row = []
    for _, a, b in PAIRS:
        rho, _ = stats.spearmanr(sub[a], sub[b])
        row += [round(rho, 6), classify(rho)]
    records.append(row); index.append(lang)

cols = pd.MultiIndex.from_tuples(
    [(lab, sub) for lab, _, _ in PAIRS for sub in ('Spearman ρ', 'Correlation')])
corr_lang = pd.DataFrame(records, index=index, columns=cols)
corr_lang.index.name = 'Language'
corr_lang.to_csv(OUTPUTS_DIR / 'per_language_correlations.csv')
print(f"Saved → {OUTPUTS_DIR / 'per_language_correlations.csv'}")

# ── paper-style table figure (grouped 2-line headers, compiler-tinted rows) ──
def _tint(hexc, amt=0.85):
    r, g, b = mcolors.to_rgb(hexc)
    return (r + (1 - r) * amt, g + (1 - g) * amt, b + (1 - b) * amt)

headers = ['Language',
           'Energy & Time\nSpearman ρ', 'Energy & Time\nCorrelation',
           'Energy & Memory\nSpearman ρ', 'Energy & Memory\nCorrelation',
           'Memory & Time\nSpearman ρ', 'Memory & Time\nCorrelation']
cell_text = []
for lang in index:
    rec = corr_lang.loc[lang]
    cell_text.append([
        lang,
        f"{rec[('Energy & Time','Spearman ρ')]:.3f}",   rec[('Energy & Time','Correlation')],
        f"{rec[('Energy & Memory','Spearman ρ')]:.3f}", rec[('Energy & Memory','Correlation')],
        f"{rec[('Memory & Time','Spearman ρ')]:.3f}",   rec[('Memory & Time','Correlation')],
    ])

fig, ax = plt.subplots(figsize=(16, 9))
ax.axis('off')
table = ax.table(cellText=cell_text, colLabels=headers, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.7)
table.auto_set_column_width(col=list(range(len(headers))))

for j in range(len(headers)):
    h = table[(0, j)]
    h.set_facecolor('#2c3e50')
    h.set_text_props(color='white', fontweight='bold')
    h.set_height(0.085)

for i, lang in enumerate(index, start=1):
    base = COMPILER_COLORS[COMPILER[lang]]
    for j in range(len(headers)):
        table[(i, j)].set_facecolor(_tint(base, 0.85))
    table[(i, 0)].set_text_props(fontweight='bold')

legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                  for p in COMPILER_ORDER]
ax.legend(handles=legend_handles, title='Execution Model', loc='lower center',
          bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=True)
ax.set_title('Per-Language Correlation Values (Spearman ρ across the 8 benchmarks, N=8)\n'
             'classification by Rea & Parker effect size', fontsize=13, pad=16)
plt.tight_layout()
ps.save_fig(fig, '04_per_language_correlations')
plt.show()
corr_lang